In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Локальний режим тестування Qiskit Runtime

<span id="test-locally"></span>


<details>
<summary><b>Версії пакетів</b></summary>

Код на цій сторінці розроблено з використанням наступних залежностей.
Рекомендуємо використовувати ці або новіші версії.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
qiskit-aer~=0.17
```
</details>
Використовуй локальний режим тестування (доступний у `qiskit-ibm-runtime` v0.22.0 або новіших версіях), щоб тестувати програми перед їх налаштуванням і відправкою на справжнє квантове залізо. Після того як ти перевіриш свою програму в локальному режимі тестування, достатньо лише змінити назву Backend, щоб запустити її на QPU.

Щоб скористатися локальним режимом тестування, вкажи один із фейкових Backend з ``qiskit_ibm_runtime.fake_provider`` або вкажи Backend Qiskit Aer під час створення примітиву Qiskit Runtime або сесії.

- **Фейкові Backend**: [Фейкові Backend](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider) у ``qiskit_ibm_runtime.fake_provider`` імітують поведінку QPU IBM&reg; за допомогою знімків QPU. Знімки QPU містять важливу інформацію про QPU: карту зв'язності, базові Gate та властивості кубітів — усе це корисно для тестування Transpiler та виконання зашумлених симуляцій QPU. Модель шуму зі знімку застосовується автоматично під час симуляції.
- **Симулятор Aer**: Симулятори з [Qiskit Aer](/guides/simulate-with-qiskit-aer) забезпечують більш продуктивну симуляцію, яка може обробляти більші Circuit та [власні моделі шуму](/guides/build-noise-models). Перелік доступних методів симуляції можна переглянути при використанні `AerSimulator` у локальному режимі тестування. Дивись [приклад режиму симуляції Кліффорда](#clifford-sim), який демонструє ефективну симуляцію Circuit Кліффорда з великою кількістю кубітів.

    <details>
    <summary>**Перелік методів симуляції, доступних у Qiskit Aer**</summary>

    Докладніше дивись у документації [`AerSimulator`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator).

    * ``"automatic"``: Метод симуляції за замовчуванням. Автоматично обирає метод симуляції залежно від Circuit та моделі шуму.

    * ``"statevector"``: Симуляція щільного вектора стану, яка може вибирати результати вимірювань із *ідеальних* Circuit з усіма вимірами в кінці Circuit. Для зашумлених симуляцій кожний знімок вибирає випадковим чином зашумлений Circuit із моделі шуму.

    * ``"density_matrix"``: Симуляція матриці густини, яка може вибирати результати вимірювань із *зашумлених* Circuit з усіма вимірами в кінці Circuit.

    * ``"stabilizer"``: Ефективний симулятор стану стабілізатора Кліффорда, який може симулювати зашумлені Circuit Кліффорда, якщо всі помилки в моделі шуму також є помилками Кліффорда.

    * ``"extended_stabilizer"``: Наближений симулятор для Circuit Кліффорда + T, що базується на розкладанні стану у ранжований стан стабілізатора. Кількість доданків зростає зі збільшенням кількості non-Clifford (T) Gate.

    * ``"matrix_product_state"``: Симулятор вектора стану на основі тензорних мереж, що використовує представлення Matrix Product State (MPS) для стану. Це можна робити як із обрізанням, так і без обрізання вимірностей зв'язку MPS залежно від параметрів симулятора. За замовчуванням обрізання не застосовується.

    * ``"unitary"``: Симуляція щільної унітарної матриці ідеального Circuit. Симулює унітарну матрицю самого Circuit, а не еволюцію початкового квантового стану. Цей метод може симулювати лише Gate; він не підтримує вимірювання, скидання або шум.

    * ``"superop"``: Симуляція щільної матриці суперопоратора ідеального або зашумленого Circuit. Симулює матрицю суперопоратора самого Circuit, а не еволюцію початкового квантового стану. Цей метод може симулювати ідеальні та зашумлені Gate і скидання, але не підтримує вимірювання.

    * ``"tensor_network"``: Симуляція на основі тензорних мереж, що підтримує як вектор стану, так і матрицю густини. Наразі доступно лише для GPU і прискорюється за допомогою API cuQuantum `cuTensorNet`.
    </details>

> **Note:** - У локальному режимі тестування можна вказати всі параметри Qiskit Runtime. Проте всі параметри, крім shots, ігноруються під час запуску на локальному симуляторі.
> - Рекомендується встановити Qiskit Aer перед використанням фейкових Backend або симуляторів Aer, виконавши `pip install qiskit-aer`. Фейкові Backend використовують симулятори Aer за наявності, щоб скористатися їхньою продуктивністю.


## Приклад із фейковими Backend

In [1]:
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit_ibm_runtime.fake_provider import FakeManilaV2

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using FakeManilaV2
fake_manila = FakeManilaV2()
pm = generate_preset_pass_manager(backend=fake_manila, optimization_level=1)
isa_qc = pm.run(qc)

# You can use a fixed seed to get fixed results.
options = {"simulator": {"seed_simulator": 42}}
sampler = Sampler(mode=fake_manila, options=options)

result = sampler.run([isa_qc]).result()

## Приклади AerSimulator
Приклад із сесіями, без шуму:

In [2]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

# Run the sampler job locally using AerSimulator.
# Session syntax is supported but ignored because local mode doesn't support sessions.
aer_sim = AerSimulator()
pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
with Session(backend=aer_sim) as session:
    sampler = Sampler(mode=session)
    result = sampler.run([isa_qc]).result()

Щоб симулювати з шумом, вкажи QPU (квантове залізо) і відправ його до Aer. Aer будує модель шуму на основі калібрувальних даних цього QPU та створює Backend Aer з цією моделлю. За бажанням ти можеш [побудувати модель шуму](/guides/build-noise-models) самостійно.

> **Caution:** QPU може зазнавати впливу різних видів шуму. Модель шуму Qiskit Aer, що використовується тут, симулює лише деякі з них, тому вона, ймовірно, буде менш значною, ніж шум на справжньому QPU.
> 
> Докладніше про те, які помилки включаються при ініціалізації моделі шуму з QPU, дивись у довіднику API Aer [`NoiseModel`](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.noise.NoiseModel.html#qiskit_aer.noise.NoiseModel.from_backend).

Приклад із шумом:

In [3]:
from qiskit_aer import AerSimulator
from qiskit.circuit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import SamplerV2 as Sampler, QiskitRuntimeService

# Bell Circuit
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

service = QiskitRuntimeService()

# Specify a QPU to use for the noise model
real_backend = service.backend("ibm_fez")
aer = AerSimulator.from_backend(real_backend)

# Run the sampler job locally using AerSimulator.
pm = generate_preset_pass_manager(backend=aer, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer)
result = sampler.run([isa_qc]).result()

<span id="clifford-sim"></span>
### Симуляція Кліффорда
Оскільки Circuit Кліффорда можна ефективно симулювати з верифікованими результатами, симуляція Кліффорда є дуже корисним інструментом. Для детального прикладу дивись [Ефективна симуляція стабілізаторних Circuit за допомогою примітивів Qiskit Aer](/guides/simulate-stabilizer-circuits).

Приклад:

In [4]:
import numpy as np
from qiskit.circuit.library import efficient_su2
from qiskit_ibm_runtime import SamplerV2 as Sampler

n_qubits = 500  # <---- note this uses 500 qubits!
circuit = efficient_su2(n_qubits)
circuit.measure_all()

rng = np.random.default_rng(1234)
params = rng.choice(
    [0, np.pi / 2, np.pi, 3 * np.pi / 2],
    size=circuit.num_parameters,
)

# Tell Aer to use the stabilizer (Clifford) simulation method
aer_sim = AerSimulator(method="stabilizer")

pm = generate_preset_pass_manager(backend=aer_sim, optimization_level=1)
isa_qc = pm.run(qc)
sampler = Sampler(mode=aer_sim)
result = sampler.run([isa_qc]).result()

## Подальші кроки
> **Tip:** - Переглянь детальні [приклади використання примітивів.](/guides/primitives-examples)
>     - Прочитай [Перехід до примітивів V2](/guides/v2-primitives).
>     - Попрактикуйся з примітивами, опрацювавши [урок про функцію вартості](/learning/courses/variational-algorithm-design/cost-functions) у IBM Quantum Learning.
>     - Дізнайся, як виконувати транспіляцію локально, у розділі [Transpile](/guides/transpile).
>     - Спробуй навчальний посібник [Порівняння налаштувань Transpiler](/guides/circuit-transpilation-settings#compare-transpiler-settings).